##### ARTI 560 - Computer Vision

## Image Classification with Vision Transformer (ViT) - Exercise 

### Objective

In this exercise, you will test the pretrained Vision Transformer (ViT) model on 5 real-world images that you find online.

You will:

1. Download 5 images for different classes in [ImageNet](https://github.com/Waikato/wekaDeeplearning4j/blob/master/docs/user-guide/class-maps/IMAGENET.md).

2. Load the ImageNet class names from a [text file](https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt).

3. Use ViT to predict the class for each image.

4. Record whether the prediction was correct.

#### Important Note

For this exercise, you MUST use the following KerasHub components:

- [keras_hub.models.ViTImageClassifier](https://keras.io/keras_hub/api/models/vit/vit_image_classifier/)

- [keras_hub.models.ViTImageClassifierPreprocessor](https://keras.io/keras_hub/api/models/vit/vit_image_classifier_preprocessor/)

This ensures your input preprocessing (resizing + normalization) matches what the pretrained ViT model expects.

Do not replace the preprocessor with manual normalization (such as dividing by 255), because it may produce incorrect predictions.

In [1]:
import os
import requests
import numpy as np
import PIL.Image as Image
import io
import time
import keras_hub

# 1. Setup Headers to look like a browser
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

# 2. Load ImageNet labels
classes_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
imagenet_classes = requests.get(classes_url).text.splitlines()

# 3. Load Model & Preprocessor (Required by Lab)
preprocessor = keras_hub.models.ViTImageClassifierPreprocessor.from_preset(
    "vit_base_patch16_224_imagenet"
)
model = keras_hub.models.ViTImageClassifier.from_preset(
    "vit_base_patch16_224_imagenet", 
    preprocessor=preprocessor
)

# 4. New, stable URLs to avoid 404/429 errors
image_links = {
    "golden_retriever.jpg": "https://images.dog.ceo/breeds/retriever-golden/n02099601_3004.jpg",
    "espresso.jpg": "https://images.pexels.com/photos/302899/pexels-photo-302899.jpeg?auto=compress&cs=tinysrgb&w=400",
    "airliner.jpg": "https://images.pexels.com/photos/46148/aircraft-jet-landing-cloud-46148.jpeg?auto=compress&cs=tinysrgb&w=400",
    "school_bus.jpg": "https://images.pexels.com/photos/209679/pexels-photo-209679.jpeg?auto=compress&cs=tinysrgb&w=400",
    "giant_panda.jpg": "https://images.pexels.com/photos/1661535/pexels-photo-1661535.jpeg?auto=compress&cs=tinysrgb&w=400"
}

def predict_image(file_name, url):
    try:
        time.sleep(1) # Small delay to prevent 429 errors
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        
        img = Image.open(io.BytesIO(response.content)).convert("RGB")
        img_array = np.array(img)
        
        # Batch dimension: (1, H, W, 3)
        img_batch = np.expand_dims(img_array, axis=0)
        
        # Model handles preprocessing internally via the preset
        predictions = model.predict(img_batch)
        predicted_idx = np.argmax(predictions, axis=-1)[0]
        
        return imagenet_classes[predicted_idx]
    except Exception as e:
        return f"Error: {e}"

# Run and Display Results
print(f"{'File Name':<20} | {'Predicted Label':<20}")
print("-" * 45)
for name, url in image_links.items():
    label = predict_image(name, url)
    print(f"{name:<20} | {label:<20}")

100%|██████████| 593/593 [00:00<00:00, 1.06MB/s]


100%|██████████| 1.65k/1.65k [00:00<00:00, 3.20MB/s]


File Name            | Predicted Label     
---------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
golden_retriever.jpg | golden retriever    
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 607ms/step
espresso.jpg         | espresso            
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 814ms/step
airliner.jpg         | airliner            
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 626ms/step
school_bus.jpg       | teddy               
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 643ms/step
giant_panda.jpg      | giant panda         


### Record Your Results

Fill the table below based on your results:

| Image File   | Predicted Label | True Label (What you searched) | Correct? (Yes/No) |
| ------------ | --------------- | ------------------------------ | ----------------- |
| golden_retriever.jpg | **golden retriever** | Golden Retriever | Yes |
| espresso.jpg		 | **espresso** | Espresso | Yes |
| airliner.jpg | **airliner** | Airliner | Yes |
| school_bus.jpg | **teddy** | School Bus | No |
| giant_panda.jpg | **giant_panda** | giant_panda | Yes |

